## Uploading file into postgres database -AWS


This notebook allow you to import all csv files in your directry into db

Steps:
- import CSV file into pandas dataframe
- clean table name and remove all extra symbols, spaces and capital letters
- clean table headers and remove all extra symbols, spaces and capital letters
- write the create table SQL statement
- import the data into database
    

In [1]:
import os
import numpy as np
import pandas as pd
import psycopg2

Finding your CSV files in your current working directory

In [4]:
csv_files = []
for file in os.listdir(os.getcwd()):
    if file.endswith('.csv'):
        csv_files.append(file)

In [5]:
csv_files

['Covid_Deaths.csv', 'Covid_Vaccinations.csv']

Making new directory (create bash command to make new dir)

In [7]:
dataset_dir = 'dataset'

try:
    mkdir = 'mkdir {0}'.format(dataset_dir)
    os.system(mkdir)
except:
    pass

 Move the CSV files into new directory

In [8]:
for csv in csv_files:
    mv_file = "mv '{0}' {1}".format(csv, dataset_dir)
    os.system(mv_file)
    print(mv_file)

mv 'Covid_Deaths.csv' dataset
mv 'Covid_Vaccinations.csv' dataset


Create pandas df from CSV files

In [9]:
data_path = os.getcwd() + '/' + dataset_dir + '/'

df = {}

for file in csv_files:
    try:
        df[file] = pd.read_csv(data_path+file, header=0, delimiter=';', on_bad_lines='skip', engine='python')
    except UnicodeDecodeError:
        df[file] = pd.read_csv(data_path+file, header=0, delimiter=';', on_bad_lines='skip', engine='python', enconding = "ISO-8859-1")
    print(file)

Covid_Deaths.csv
Covid_Vaccinations.csv


Clean table names and headers

In [15]:
for k in csv_files:
    dataframe = df[k]
    
    clean_tbl_name = k.lower().strip().replace(" ","_").replace("?","").replace("-","_").replace(r"/","") \
                    .replace("$","").replace("\\","").replace("&","_").replace(")","") \
                    .replace("(","").replace("%","").replace(r"(","")

    tbl_name = '{0}'.format(clean_tbl_name.split('.')[0])
    
    dataframe.columns = [x.lower().strip().replace(" ","_").replace("?","").replace("-","_").replace(r"/","") \
                    .replace("$","").replace("\\","").replace("&","_").replace(")","") \
                    .replace("(","").replace("%","").replace(r"(","") for x in dataframe.columns]
    
#     print(tbl_name)

# replace dict that maps pandas dtype with sql dtypes
    replacements = {
    'object': 'varchar',
    'float64': 'float',
    'int64': 'int',
    'datetime64': 'timestamp',
    'timedelta64[ns]': 'varchar'
    }
    
    col_str = ", ".join("{} {}".format(n, d) for (n, d) in zip(dataframe.columns, dataframe.dtypes.replace(replacements)))
    
    host='c-project.cssudyzipljs.eu-central-1.rds.amazonaws.com'
#     port='5433'
    dbname='postgres'
    user='joanna'
    password='Four-leafClover'

    conn_str = "host=%s dbname=%s user=%s password=%s" % (host, dbname, user, password)

    conn = psycopg2.connect(conn_str)
    cursor = conn.cursor()
    print('Opened database connection')
    
    cursor.execute("drop table if exists %s;" % (tbl_name))
    
    cursor.execute("create table %s (%s)" % (tbl_name, col_str))
    print('{0} was created successfully'.format(tbl_name))
    
#  insert values into table
#  save df to csv
    dataframe.to_csv(k,  header=dataframe.columns, index=False, encoding='utf-8')

#  open csv file, save it as object and upload to db
    my_file = open(k)
    print('File opened in memory')
    
#  upload csv to db
    SQL_STATEMENT = """
    COPY %s FROM STDIN WITH
        CSV
        HEADER
        DELIMITER AS ','
    """

    cursor.copy_expert(sql=SQL_STATEMENT % tbl_name, file=my_file)
    print('File copied to db')
    
#  opening acces for multiple users
    cursor.execute("grant select on table %s to public" % tbl_name)
    conn.commit()
    cursor.close()
    print('Table %s imported to db' % tbl_name)
    
# for loop end message
print('All tables have been upload into db successfully')

Opened database connection
covid_deaths was created successfully
File opened in memory
File copied to db
Table covid_deaths imported to db
Opened database connection
covid_vaccinations was created successfully
File opened in memory
File copied to db
Table covid_vaccinations imported to db
All tables have beed upload into db successfully
